# Research agent — Browserless × LangChain (Python)

Multi-step research workflow using Browserless's stateless tools: search the web with `browserless_search`, then scrape the most relevant results with `browserless_smartscraper`, then summarize. The LLM picks which tool to call at each step.

All tools used here are stateless single-shot calls — no session affinity required.

In [ ]:
%pip install -q langchain-mcp-adapters langgraph langchain-anthropic

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_anthropic import ChatAnthropic

client = MultiServerMCPClient({
    "browserless": {
        "transport": "http",
        "url": "https://mcp.browserless.io/mcp",
        "headers": {"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    }
})

tools = await client.get_tools()
stateless_tools = [t for t in tools if t.name not in ("browserless_agent", "browserless_skill")]

agent = create_react_agent(
    ChatAnthropic(model="claude-sonnet-4-6"),
    stateless_tools,
)

In [ ]:
prompt = (
    "Research the current state of WebAssembly outside the browser. "
    "Use browserless_search to find 3 recent authoritative articles, then use browserless_smartscraper "
    "to read the most relevant one and summarize the key takeaways in 5 bullet points. "
    "Cite the URL for each takeaway."
)

out = await agent.ainvoke({"messages": [{"role": "user", "content": prompt}]})
print(out["messages"][-1].content)